# Grid 4×4 — Paper reproduction & realism benchmark (seed 42)

Single notebook for **LibSignal Table 8 reproduction** on `sumo4x4` and all **realism toggles** after the grid4x4 TLS fix (PR #6).

**Scope:** seed 42 only; latest complete 200-episode `*_DTL.log` per method (tie-break: newest filename). Baselines: `*_BRF.log` Final Travel Time.

**Separate experiments:** `results.ipynb` (historical notes) · `demand_bag_1200_dqn.ipynb` (`feat/demand-bag-1200` branch only).


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline

REPO = Path('..').resolve()
OUT = REPO / 'data' / 'output_data' / 'tsc'
COLS = ['model', 'split', 'episode', 'travel_time', 'col5', 'reward', 'queue', 'delay', 'throughput']

PAPER_REPRO = {
    'rl': {
        'DQN': OUT / 'sumo_dqn' / 'sumo4x4' / 'dqn_grid4x4_seed42' / 'logger',
        'PressLight': OUT / 'sumo_presslight' / 'sumo4x4' / 'grid_repro' / 'logger',
        'CoLight': OUT / 'sumo_colight' / 'sumo4x4' / 'grid_repro' / 'logger',
        'MPLight': OUT / 'sumo_mplight' / 'sumo4x4' / 'grid_repro' / 'logger',
    },
    'baselines': {
        'MaxPressure': {'att': 155.39, 'log': 'results.ipynb probe (pre-fix net)'},
        'FixedTime': {'att': 226.74, 'log': 'results.ipynb probe (pre-fix net)'},
    },
    'paper': {
        'MaxPressure': 154.15, 'FixedTime': 222.13, 'DQN': 149.09,
        'PressLight': 152.81, 'CoLight': 154.10, 'MPLight': 147.19,
    },
    'notes': {
        'MPLight': 'fork variant in repo — not a clean paper reproduction',
        'DQN': 'IDQN in paper Table 8',
    },
}

EXPERIMENTS = {
    'homo': {
        'toggle': 'Homo — corrected grid4x4, default vType',
        'rl': {
            'DQN': OUT / 'sumo_dqn_sumo4x4' / 'sumo4x4' / 'homo_4x4' / 'logger',
            'PressLight': OUT / 'sumo_presslight_sumo4x4' / 'sumo4x4' / 'homo_4x4' / 'logger',
            'CoLight': OUT / 'sumo_colight' / 'sumo4x4' / 'homo_4x4' / 'logger',
        },
        'baselines': {
            'MaxPressure': OUT / 'sumo_maxpressure' / 'sumo4x4' / 'baseline_homo' / 'logger',
            'FixedTime': OUT / 'sumo_fixedtime' / 'sumo4x4' / 'baseline_homo' / 'logger',
        },
    },
    'slow_start': {
        'toggle': 'Slow-start — tau/actionStepLength realism (Jul 14 runs)',
        'rl': {
            'DQN': OUT / 'sumo_dqn_slow_start' / 'sumo4x4' / 'slow_start_4x4' / 'logger',
            'PressLight': OUT / 'sumo_presslight_slow_start' / 'sumo4x4' / 'slow_start_4x4' / 'logger',
            'CoLight': OUT / 'sumo_colight_slow_start' / 'sumo4x4' / 'slow_start_4x4' / 'logger',
        },
        'baselines': {
            'MaxPressure': OUT / 'sumo_maxpressure_slow_start' / 'sumo4x4' / 'baseline_slow_start' / 'logger',
            'FixedTime': OUT / 'sumo_fixedtime_slow_start' / 'sumo4x4' / 'baseline_slow_start' / 'logger',
        },
    },
    'hetero': {
        'toggle': 'Hetero — 80% car / 20% truck (Jul 14 hetero_test runs)',
        'rl': {
            'DQN': OUT / 'sumo_dqn_hetero' / 'sumo4x4' / 'hetero_test' / 'logger',
            'PressLight': OUT / 'sumo_presslight_hetero' / 'sumo4x4' / 'hetero_test' / 'logger',
            'CoLight': OUT / 'sumo_colight_hetero' / 'sumo4x4' / 'hetero_test' / 'logger',
        },
        'baselines': {
            'MaxPressure': OUT / 'sumo_maxpressure_hetero' / 'sumo4x4' / 'baseline_hetero' / 'logger',
            'FixedTime': OUT / 'sumo_fixedtime_hetero' / 'sumo4x4' / 'baseline_hetero' / 'logger',
        },
    },
    'pobs_pen': {
        'toggle': 'Partial obs — lane dropout penalty p=0.8 (Jul 14; control = homo full obs)',
        'rl': {
            'DQN': OUT / 'sumo_dqn_pobs' / 'sumo4x4' / 'pen_s42' / 'logger',
            'PressLight': OUT / 'sumo_presslight_pobs' / 'sumo4x4' / 'pen_s42' / 'logger',
            'CoLight': OUT / 'sumo_colight_pobs' / 'sumo4x4' / 'pen_s42' / 'logger',
        },
        'baselines': {
            'MaxPressure': OUT / 'sumo_maxpressure_pobs' / 'sumo4x4' / 'pen_s42' / 'logger',
            'FixedTime': OUT / 'sumo_fixedtime_pobs' / 'sumo4x4' / 'pen_s42' / 'logger',
        },
    },
    'pobs_gauss': {
        'toggle': 'Partial obs — Gaussian noise σ=2 (Jul 14; control = homo full obs)',
        'rl': {
            'DQN': OUT / 'sumo_dqn_pobs' / 'sumo4x4' / 'gauss_s42' / 'logger',
            'PressLight': OUT / 'sumo_presslight_pobs' / 'sumo4x4' / 'gauss_s42' / 'logger',
            'CoLight': OUT / 'sumo_colight_pobs' / 'sumo4x4' / 'gauss_s42' / 'logger',
        },
        'baselines': {
            'MaxPressure': OUT / 'sumo_maxpressure_pobs' / 'sumo4x4' / 'gauss_s42' / 'logger',
            'FixedTime': OUT / 'sumo_fixedtime_pobs' / 'sumo4x4' / 'gauss_s42' / 'logger',
        },
    },
}

TOGGLE_ORDER = ['homo', 'slow_start', 'hetero', 'pobs_pen', 'pobs_gauss']


def load_dtl(logger_dir: Path):
    best, best_ep, best_path = None, -1, ''
    for p in sorted(logger_dir.glob('*_DTL.log')):
        df = pd.read_csv(p, sep='\t', header=None, names=COLS)
        mx = int(df['episode'].max())
        if mx > best_ep or (mx == best_ep and p.name > best_path):
            best, best_ep, best_path = df, mx, p.name
    if best is None:
        raise FileNotFoundError(f'No DTL logs in {logger_dir}')
    return best, logger_dir / best_path


def parse_brf_att(path: Path) -> float:
    m = re.search(r'Final Travel Time is ([0-9.]+)', path.read_text())
    if not m:
        raise ValueError(f'No Final Travel Time in {path}')
    return float(m.group(1))


def load_baseline(spec):
    if isinstance(spec, dict) and 'att' in spec:
        return spec['att'], spec['log']
    brf = sorted(spec.glob('*_BRF.log'))[-1]
    return parse_brf_att(brf), str(brf.relative_to(REPO))


def make_table(cfg: dict) -> pd.DataFrame:
    rows = []
    for name, path in cfg.get('rl', {}).items():
        df, log_path = load_dtl(path)
        te = df[df.split == 'TEST']
        rows.append({
            'method': name,
            'best_test_att': round(te.travel_time.min(), 2),
            'final_test_att': round(te.travel_time.iloc[-1], 2),
            'ep_max': int(df.episode.max()),
            'log': log_path.name,
        })
    for name, spec in cfg.get('baselines', {}).items():
        att, src = load_baseline(spec)
        rows.append({
            'method': name,
            'best_test_att': round(att, 2),
            'final_test_att': round(att, 2),
            'ep_max': 'BRF',
            'log': Path(src).name if '/' in str(src) else src,
        })
    return pd.DataFrame(rows).sort_values('final_test_att').reset_index(drop=True)


def load_toggle_rl(toggle: str):
    out = {}
    for name, path in EXPERIMENTS[toggle]['rl'].items():
        df, log_path = load_dtl(path)
        out[name] = {
            'train': df[df.split == 'TRAIN'].reset_index(drop=True),
            'test': df[df.split == 'TEST'].reset_index(drop=True),
            'log': log_path,
        }
    return out


def plot_rl_curves(rl_data, title: str):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
    for ax, name in zip(axes, rl_data):
        tr, te = rl_data[name]['train'], rl_data[name]['test']
        ax.plot(tr.episode, tr.travel_time, color='#4C72B0', linestyle='--', alpha=0.8, label='train')
        ax.plot(te.episode, te.travel_time, color='#C44E52', label='test')
        ax.set_title(name)
        ax.set_xlabel('episode')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('ATT (s)')
    axes[-1].legend()
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


## 1. LibSignal paper reproduction (original network)

Reproduction of **Table 8 Grid4×4 / SUMO** on the original `grid4x4` network (`grid_repro` / `dqn_grid4x4_seed42`, seed 42, 200 episodes). Best test episode ATT vs paper.

> **Network definition was wrong.** Pre-fix `grid4x4.net.xml` (commit `ab3d7fe`) placed **`s` on through and left lanes** — see `docs/SIGNAL_CONTROL_THEORY.md` §6. Reproduction matched the paper, but those numbers are **not valid** for realism work. Network corrected in PR #6 (valid NEMA phases).


In [ ]:
rows = []
for method in ['FixedTime', 'MaxPressure', 'DQN', 'PressLight', 'CoLight', 'MPLight']:
    if method in PAPER_REPRO['baselines']:
        ours, log = load_baseline(PAPER_REPRO['baselines'][method])
    else:
        df, p = load_dtl(PAPER_REPRO['rl'][method])
        ours = round(df[df.split == 'TEST'].travel_time.min(), 2)
        log = p.name
    pval = PAPER_REPRO['paper'][method]
    rows.append({
        'method': method,
        'paper_sumo': pval,
        'ours_best_test': ours,
        'diff_vs_paper_%': round(100 * (ours - pval) / pval, 1),
        'log': log,
        'note': PAPER_REPRO['notes'].get(method, ''),
    })
paper_table = pd.DataFrame(rows)
paper_table


## 2. Corrected network — homo baselines (post TLS fix)

3600-step eval on corrected `grid4x4`, seed 42:

| Method | Final ATT (s) |
|--------|---------------|
| **MaxPressure** | **173.26** |
| **FixedTime** | **219.14** |

Slow-start, hetero, and partial-obs RL rows below use **Jul 14** reruns on this corrected network.


In [ ]:
make_table({'baselines': EXPERIMENTS['homo']['baselines']})


## 3. Realism toggles — one table per axis (seed 42, latest logs)

**DQN · PressLight · CoLight · MaxPressure · FixedTime** per toggle. Partial-obs control = homo (§2).

**Order:** Homo → Slow-start → Hetero → Partial obs (pen, gauss).


### Homo — corrected grid4x4, default vType


In [ ]:
make_table(EXPERIMENTS['homo'])


### Slow-start — tau/actionStepLength realism (Jul 14 runs)


In [ ]:
make_table(EXPERIMENTS['slow_start'])


### Hetero — 80% car / 20% truck (Jul 14 hetero_test runs)


In [ ]:
make_table(EXPERIMENTS['hetero'])


### Partial obs — lane dropout penalty p=0.8 (Jul 14; control = homo full obs)


In [ ]:
make_table(EXPERIMENTS['pobs_pen'])


### Partial obs — Gaussian noise σ=2 (Jul 14; control = homo full obs)


In [ ]:
make_table(EXPERIMENTS['pobs_gauss'])


## 4. Training curves (train dashed · test solid)


In [ ]:
homo_rl = load_toggle_rl('homo')
plot_rl_curves(homo_rl, EXPERIMENTS['homo']['toggle'])


In [ ]:
slow_rl = load_toggle_rl('slow_start')
plot_rl_curves(slow_rl, EXPERIMENTS['slow_start']['toggle'])


In [ ]:
hetero_rl = load_toggle_rl('hetero')
plot_rl_curves(hetero_rl, EXPERIMENTS['hetero']['toggle'])


In [ ]:
pobs_pen_rl = load_toggle_rl('pobs_pen')
plot_rl_curves(pobs_pen_rl, EXPERIMENTS['pobs_pen']['toggle'])


In [ ]:
pobs_gauss_rl = load_toggle_rl('pobs_gauss')
plot_rl_curves(pobs_gauss_rl, EXPERIMENTS['pobs_gauss']['toggle'])


## Summary

- **§1** Paper repro on **wrong** network — gate passed, then invalidated by TLS audit.
- **§2** Corrected homo baselines: MP **173.26 s**, FT **219.14 s**.
- **§3–§4** One table + curves per realism toggle; Jul 14 logs for slow-start / hetero / partial-obs.

**Replaces** `hetero_homo_4x4_rl.ipynb` + old `realism_benchmark_4x4.ipynb`.
